In [135]:
import polars as pl
from polars import col


pl.Config.set_tbl_rows(1000)      # Allow rendering up to 1000 rows inline
pl.Config.set_tbl_cols(20)        # Allow rendering up to 20 columns wide
pl.Config.set_fmt_str_lengths(50) # Prevent text strings from getting cropped



polars.config.Config

In [136]:
market_response_path = "/Users/oceansxyz/alphanode-metal/logs/market/market_20260616_155322.jsonl"

#### Load JSONL - Binace Market WS

In [137]:
#Raw binance reponse for market web socket - pretty much a static
book_ticker_schema = pl.Struct([
    pl.Field("ts", pl.Int64),
    pl.Field("frame", pl.Struct([
        pl.Field("u", pl.Int64),
        pl.Field("s", pl.String),
        pl.Field("b", pl.String),
        pl.Field("B", pl.String),
        pl.Field("a", pl.String),
        pl.Field("A", pl.String),
    ])),
])

with open(market_response_path) as f:
    metadata_json = f.readline().strip()
    print("Metadata Raw Line:", metadata_json)
    df_market = (
        pl.read_csv(
            f,
            has_header=False,
            new_columns=["json_str"],
            separator="\n",
            infer_schema=False,
        )
        .select(pl.col("json_str").str.json_decode(dtype=book_ticker_schema))
        .unnest("json_str")
        .unnest("frame")
        .with_columns([
            pl.col("u").cast(pl.Int64),
            pl.col("s"),
            pl.col("b").cast(pl.Float64),
            pl.col("B").cast(pl.Float64),
            pl.col("a").cast(pl.Float64),
            pl.col("A").cast(pl.Float64),
        ])
    )


# Select relevant cols for alpha calulation with identifiable file names
market_rates = df_market.select(
    col("ts"), col("s").alias("symbol"), col("b").alias("bid"), col("a").alias("ask")
)

# duration of web socket listen
min_ts = market_rates.min().select("ts").item()
max_ts = market_rates.max().select("ts").item()
duration_m =(max_ts-min_ts)/60000


print(f"Duration of listen : {duration_m} mins")
print(market_rates.group_by("symbol").len())

Metadata Raw Line: {"ts":1781625203560,"frame":{"result":null,"id":1}}
Duration of listen : 8.21975 mins
shape: (3, 2)
┌──────────┬──────┐
│ symbol   ┆ len  │
│ ---      ┆ ---  │
│ str      ┆ u32  │
╞══════════╪══════╡
│ BTCUSDT  ┆ 5843 │
│ BTCEURI  ┆ 68   │
│ EURIUSDT ┆ 237  │
└──────────┴──────┘


In [138]:
market_rates.head()

ts,symbol,bid,ask
i64,str,f64,f64
1781625203562,"""BTCUSDT""",65958.26,65958.27
1781625203602,"""BTCUSDT""",65958.26,65958.28
1781625203602,"""BTCUSDT""",65958.26,65958.43
1781625203602,"""BTCUSDT""",65958.26,65958.44
1781625203602,"""BTCUSDT""",65958.26,65959.28


In [139]:
## Split dataframe symbolwise symbol is an orderbook
symbol_0 = market_rates.filter(col("symbol")=="BTCUSDT").select(["ts", col("bid").alias("BTCUSDT_bid"), col("ask").alias("BTCUSDT_ask")])
symbol_1 = market_rates.filter(col("symbol")=="BTCEURI").select(["ts", col("bid").alias("BTCEURI_bid"), col("ask").alias("BTCEURI_ask")])
symbol_2 = market_rates.filter(col("symbol")=="EURIUSDT").select(["ts", col("bid").alias("EURIUSDT_bid"), col("ask").alias("EURIUSDT_ask")])

In [140]:
# Forward fill missing values by copying the last known valid price down 
# for all generated ticker metrics simultaneously, skipping the timestamp.
market_rates_widened = symbol_0.join(symbol_1, on="ts", how="full", coalesce=True)\
    .join(symbol_2, on="ts", how="full", coalesce=True)\
    .sort("ts")\
    .select([
        col("ts"),
        col("*").exclude("ts").fill_null(strategy="forward")
    ]).drop_nulls()

market_rates_widened.head()

ts,BTCUSDT_bid,BTCUSDT_ask,BTCEURI_bid,BTCEURI_ask,EURIUSDT_bid,EURIUSDT_ask
i64,f64,f64,f64,f64,f64,f64
1781625212309,65958.0,65958.01,56660.47,56808.94,1.1613,1.1616
1781625212339,65958.0,65958.01,56660.47,56808.94,1.1613,1.1616
1781625212593,65957.99,65958.01,56660.47,56808.94,1.1613,1.1616
1781625212593,65957.4,65958.01,56660.47,56808.94,1.1613,1.1616
1781625212593,65957.98,65958.01,56660.47,56808.94,1.1613,1.1616


#### Notation & Model

NODE_0 = USDT NODE_1 = BTC NODE_2 = EURI  

EDGE_0 = BTCUSDT  EDGE_1 = BTCEURI  EDGE_2 = EURIUSDT  

Clockwise Walk  0→1  →1→2  →2→0  

BUY  BTCUSDT  @ASK  === 0→1  
SELL BTCEURI  @BID  === 1→2  
SELL EURIUSDT @BID  === 2→0


#### Clockwise Walk  0→1  →1→2  →2→0

In [141]:
clockwise_walk = market_rates_widened.select("ts", col("BTCUSDT_ask"), "BTCEURI_bid", "EURIUSDT_bid")\
    .unique(subset=["BTCUSDT_ask", "BTCEURI_bid", "EURIUSDT_bid"])\
    .with_columns((col("BTCEURI_bid")*col("EURIUSDT_bid")).alias("BTCEURIUSDT_bid"))\
    .with_columns((col("BTCEURIUSDT_bid")/col("BTCUSDT_ask")).abs().alias("alpha"))\
    .with_columns(((col("alpha")-1)/(1e-4)).alias("alpha_bps"))



clockwise_walk.sort("alpha", descending=True).head(20)



ts,BTCUSDT_ask,BTCEURI_bid,EURIUSDT_bid,BTCEURIUSDT_bid,alpha,alpha_bps
i64,f64,f64,f64,f64,f64,f64
1781625598677,65859.76,56660.82,1.1616,65817.208512,0.999354,-6.460924
1781625601607,65859.77,56660.82,1.1616,65817.208512,0.999354,-6.462441
1781625601607,65859.78,56660.82,1.1616,65817.208512,0.999354,-6.463958
1781625601607,65859.99,56660.82,1.1616,65817.208512,0.99935,-6.495824
1781625601607,65860.0,56660.82,1.1616,65817.208512,0.99935,-6.497341
1781625601607,65860.4,56660.82,1.1616,65817.208512,0.999344,-6.558036
1781625601607,65860.5,56660.82,1.1616,65817.208512,0.999343,-6.57321
1781625601607,65861.98,56660.82,1.1616,65817.208512,0.99932,-6.797774
1781625601607,65861.99,56660.82,1.1616,65817.208512,0.99932,-6.799292


#### AntiClockwise Walk   0←1 ←1←2  ←2←0

Remarkably profitable - SELL BTCUSDT; BUY BTCEURI; and BUY EURIUSDT

In [142]:
anticlockwise_walk = market_rates_widened.select("ts", col("BTCUSDT_bid"), "BTCEURI_ask", "EURIUSDT_ask")\
    .unique(subset=["BTCUSDT_bid", "BTCEURI_ask", "EURIUSDT_ask"])\
    .with_columns((col("BTCEURI_ask")*col("EURIUSDT_ask")).alias("BTCEURIUSDT_ask"))\
    .with_columns((col("BTCEURIUSDT_ask")/col("BTCUSDT_bid")).abs().alias("alpha"))\
    .with_columns(((col("alpha")-1)/(1e-4)).alias("alpha_bps"))



anticlockwise_walk.sort("alpha", descending=True).head(20)

ts,BTCUSDT_bid,BTCEURI_ask,EURIUSDT_ask,BTCEURIUSDT_ask,alpha,alpha_bps
i64,f64,f64,f64,f64,f64,f64
1781625598243,65857.36,56783.34,1.1617,65965.206078,1.001638,16.375706
1781625532313,65879.79,56788.0,1.162,65987.656,1.001637,16.373155
1781625532090,65880.77,56788.0,1.162,65987.656,1.001622,16.224158
1781625514052,65910.0,56792.05,1.1624,66015.07892,1.001594,15.942789
1781625531611,65885.37,56788.0,1.162,65987.656,1.001552,15.524843
1781625598321,65857.36,56778.3,1.1617,65959.35111,1.001549,15.486668
1781625532594,65886.02,56788.0,1.162,65987.656,1.001543,15.426034
1781625527608,65887.7,56788.0,1.162,65987.656,1.001517,15.170662
1781625527608,65887.72,56788.0,1.162,65987.656,1.001517,15.167622


In [47]:
anticlockwise_walk.filter((col("ts")<1781435059000) & (col("ts")>1781435057000))\
    .plot.line(x="ts", y="BTCUSDT_bid")

alt.Chart(...)